<a href="https://colab.research.google.com/github/Jstrad99/Fake-News-Detection/blob/main/Final_team_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Technical Report: Fake News Detection Using NLP and Machine Learning


In [ ]:
## import and install libraries
!pip install pandas numpy scikit-learn nltk tensorflow

import pandas as pd
import numpy as np
import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
!pip install kagglehub

## Load Dataset

In [ ]:
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("sumanthvrao/fakenewsdataset")
print("Path to dataset files:", path)
print("Contents of dataset directory:", os.listdir(path))
print("Contents of overall directory:", os.listdir(os.path.join(path, 'overall')))
# used google gemini for function to read all text files from a given directory
# and assign a label
def load_news_from_dir(directory, label):
    data = []
    for filename in os.listdir(directory):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory, filename)
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    content = f.read()
                    data.append({'text': content, 'label': label})
            except Exception as e:
                print(f"Error reading {filepath}: {e}")
    return pd.DataFrame(data)
# Define the paths to the fake and real news
fake_news_dir = os.path.join(path, 'overall', 'overall', 'fake')
true_news_dir = os.path.join(path, 'overall', 'overall', 'real')
# Load fake news (label 0)
fake = load_news_from_dir(fake_news_dir, 0)
# Load real news (label 1)
real = load_news_from_dir(true_news_dir, 1)
df = pd.concat([fake, real])
df = df.sample(frac=1).reset_index(drop=True)
df['content'] = df['text'] + '' + df['text']
print("Successfully loaded data. DataFrame shape:", df.shape)
print("First 5 rows of the DataFrame:")
print(df.head())

In [ ]:
import kagglehub

path = kagglehub.dataset_download("sumanthvrao/fakenewsdataset")
print("Path to dataset files:", path)

## Data Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z\s]', '', text) # Keep spaces
  text = text.split()
  text = [word for word in text if word not in stop_words]
  text = ' '.join(text)
  return text # Return the correctly joined string

df['text'] = df['text'].apply(clean_text)
df['content'] = df['content'].apply(clean_text)

## Train and Test

In [ ]:
X = df['content']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)
display(X_train.head())
display(y_train.head())

## TF-TDF Vecteization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# used google gemini to assist in changing max features for better results
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,2), min_df=2)
# Fit the vectorizer only on the training data to prevent data leakage
vectorizer.fit(X_train)
X_train_tfidf = vectorizer.transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

## Train ML Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import matplotlib.pyplot as plt

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred = lr.predict(X_test_tfidf)
print("Logistic Regression:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
feature_names = vectorizer.get_feature_names_out()
coefficients = lr.coef_[0]

top_real = np.argsort(coefficients)[-10:]
top_fake = np.argsort(coefficients)[:10]

top_features = np.concatenate([top_fake, top_real])
top_words = feature_names[top_features]
top_scores = coefficients[top_features]

plt.figure()
plt.barh(top_words, top_scores)
plt.title("Top Features for Fake vs Real News")
plt.xlabel("Coefficient Value")
plt.show()

## Logistic regression with class weights

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# used google gemini for syntax to compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, class_weights))

# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)
lr.fit(X_train_tfidf, y_train)

# Get predicted probabilities for the test set
y_probs = lr.predict_proba(X_test_tfidf)[:, 1]

from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Probability Distribution

In [ ]:
plt.figure()
plt.hist(y_probs, bins=20)
plt.title("Prediction Probability Distribution")
plt.xlabel("Predicted Probability (Real)")
plt.ylabel("Frequency")
plt.show()

## Percision Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, y_probs)

plt.figure()
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

## Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

## Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix for Logistic Regression
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix (Logistic Regression)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Feature Importance

In [ ]:
feauture_names = vectorizer.get_feature_names_out()
coefficients = lr.coef_[0]
top_fake = np.argsort(coefficients)[:10]
top_real = np.argsort(coefficients)[-10:]
print("Words associated with fake news:")
for i in top_fake:
  print(feauture_names[i])
print("\nWords associated with real news:")
for i in top_real:
  print(feauture_names[i])

## Deep Learning LSTM Model

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=200)
X_test_pad = pad_sequences(X_test_seq, maxlen=200)
# build LSTM
model = Sequential()
model.add(Embedding(15000, 128))
model.add(LSTM(128))   # increase from 64 → 128
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()
# train/evaluate
history = model.fit(X_train_pad, y_train, epochs=10, batch_size=64, validation_split=0.2,
class_weight=class_weights_dict)
val_accuracy = history.history['val_accuracy'][-1]
print(f"LSTM Validation Accuracy: {val_accuracy:.4f}")

## Sensitivity Analysis

In [ ]:
test_samples = [
    # Realistic REAL news
    "The Senate passed a $1.2 trillion infrastructure bill after months of negotiations and bipartisan support in Congress.",
    "Scientists published a peer-reviewed study in a major journal confirming new climate change trends based on long-term data analysis.",

    # Realistic FAKE news
    "BREAKING: Secret government conspiracy revealed as insiders claim hidden alien technology is being used worldwide.",
    "Viral reports claim a miracle cure for cancer has been discovered but is being suppressed by major pharmaceutical companies."]

threshold = 0.45

# Logistic Regression
test_tfidf = vectorizer.transform(test_samples)
lr_probs = lr.predict_proba(test_tfidf)

print("\nLogistic Regression Predictions:")
for text, prob in zip(test_samples, lr_probs):
    label = "Real" if prob[1] > threshold else "Fake"
    print(f"{text} -> {label} ({prob[1]:.4f})")


# LSTM
test_seq = tokenizer.texts_to_sequences(test_samples)
test_pad = pad_sequences(test_seq, maxlen=200)
lstm_preds = model.predict(test_pad)

print("\nLSTM Predictions:")
for text, pred in zip(test_samples, lstm_preds):
    label = "Real" if pred[0] > threshold else "Fake"
    print(f"{text} -> {label} ({pred[0]:.4f})")

In [ ]:
from transformers import pipeline
from google.colab import userdata
import os

# Retrieve the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Configure the Hugging Face environment to use the token
# This makes the token available to the transformers library and other Hugging Face tools
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("Hugging Face token successfully loaded from secrets and set in environment variables.")
else:
    print("Hugging Face token not found in secrets. Please ensure it's named 'HF_TOKEN'.")

# Initialize a text classification pipeline from Hugging Face
hf_classifier = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english')

# Test sentences
test_samples = [
    "The Senate passed a $1.2 trillion infrastructure bill after months of negotiations.",
    "Scientists published a peer-reviewed study confirming new climate change trends.",
    "BREAKING: Secret government conspiracy revealed as insiders claim hidden alien technology is being used worldwide.",
    "Viral reports claim a miracle cure for cancer has been discovered but suppressed by major pharmaceutical companies."]

# Logistic Regression predictions
lr_preds_labels = ['Real' if pred==1 else 'Fake' for pred in lr.predict(vectorizer.transform(test_samples))]
lr_preds_probs = [f"{prob[1]:.4f}" for prob in lr.predict_proba(vectorizer.transform(test_samples))]

# LSTM predictions
test_seq = tokenizer.texts_to_sequences(test_samples)
test_pad = pad_sequences(test_seq, maxlen=200)
lstm_preds_probs = [pred[0] for pred in model.predict(test_pad)]
lstm_preds_labels = ['Real' if p>0.5 else 'Fake' for p in lstm_preds_probs]
lstm_preds_probs = [f"{p:.4f}" for p in lstm_preds_probs]

# Hugging Face predictions
hf_results = []
for text in test_samples:
    prediction_dict = hf_classifier(text)[0] # Get the single prediction dictionary
    hf_results.append((prediction_dict['label'], f"{prediction_dict['score']:.4f}"))

# Combine all into a DataFrame
import pandas as pd

df_results = pd.DataFrame({
    "Text": test_samples,
    "Logistic Regression": lr_preds_labels,
    "LR Probability": lr_preds_probs,
    "LSTM": lstm_preds_labels,
    "LSTM Probability": lstm_preds_probs,
    "Hugging Face": [label for label, _ in hf_results],
    "HF Probability": [prob for _, prob in hf_results]})

### Hugging Face Pipeline

In [ ]:
from transformers import pipeline
# Initialize a text classification pipeline from Hugging Face
# Using a pre-trained model for sentiment analysis or fake news detection (e.g., 'distilbert-base-uncased-finetuned-sst-2-english')
hf_classifier = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english')

In [ ]:
# Retrieve the Hugging Face token from Colab secrets
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')

# Configure the Hugging Face environment to use the token
# This makes the token available to the transformers library and other Hugging Face tools
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("Hugging Face token successfully loaded from secrets and set in environment variables.")
else:
    print("Hugging Face token not found in secrets. Please ensure it's named 'HF_TOKEN'.")

## Data Preperation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Test sentences
test_samples = [
    "The Senate passed a $1.2 trillion infrastructure bill after months of negotiations.",
    "Scientists published a peer-reviewed study confirming new climate change trends.",
    "BREAKING: Secret government conspiracy revealed as insiders claim hidden alien technology is being used worldwide.",
    "Viral reports claim a miracle cure for cancer has been discovered but suppressed by major pharmaceutical companies."]

# Logistic Regression probabilities for Real class
lr_probs = lr.predict_proba(vectorizer.transform(test_samples))[:, 1]

# LSTM probabilities
test_seq = tokenizer.texts_to_sequences(test_samples)
test_pad = pad_sequences(test_seq, maxlen=200)
lstm_probs = model.predict(test_pad).flatten()

# Hugging Face probabilities for REAL label
hf_probs = [hf_classifier(text)[0]['score'] for text in test_samples]

## Comparison Plots

In [ ]:
index = np.arange(len(test_samples))
bar_width = 0.2

plt.figure(figsize=(6, 4))

plt.bar(index - bar_width, lr_probs, bar_width, label='Logistic Regression', color='skyblue')
plt.bar(index, lstm_probs, bar_width, label='LSTM', color='lightcoral')
plt.bar(index + bar_width, hf_probs, bar_width, label='Hugging Face (Sentiment)', color='lightgreen')

plt.xlabel('Test Sample')
plt.ylabel('Probability of Real News / Positive Sentiment')
plt.title('Comparison of Model Predictions for Test Samples')
plt.xticks(index, [f'Sample {i+1}' for i in range(len(test_samples))], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

The bar chart above visually compares the probabilities from Logistic Regression, LSTM, and the Hugging Face (sentiment) model for each test sample.

In [ ]:
x = np.arange(len(test_samples))  # positions for each sentence
width = 0.25  # width of each bar
plt.figure(figsize=(6,4))
plt.bar(x - width, lr_probs, width, label='Logistic Regression', color='skyblue')
plt.bar(x, lstm_probs, width, label='LSTM', color='orange')
plt.bar(x + width, hf_probs, width, label='Hugging Face', color='green')
plt.xticks(x, [f"Sentence {i+1}" for i in range(len(test_samples))], rotation=15)
plt.ylabel("Probability of Real News")
plt.title("Comparison of Model Predictions (Real News Probability)")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()